In [1]:
import pandas as pd
import numpy as np

In [2]:
sales = pd.read_csv("../data/processed/store_daily_sales.csv")

sales["Date"] = pd.to_datetime(sales["Date"])

sales = sales.sort_values(["Store", "Date"])

sales.head()

,Store,Date,Sales
0,1,2013-01-02,5530
1,1,2013-01-03,4327
2,1,2013-01-04,4486
3,1,2013-01-05,4997
4,1,2013-01-07,7176


In [3]:
raw = pd.read_csv("../data/raw/train.csv")

raw["Date"] = pd.to_datetime(raw["Date"])

raw["StateHoliday"] = raw["StateHoliday"].astype(str)

context = raw[
    ["Store", "Date", "Promo", "SchoolHoliday", "StateHoliday"]
].drop_duplicates(
    ["Store", "Date"]
)

context.head()

C:\Users\Shrey\AppData\Local\Temp\ipykernel_10000\224917563.py:1: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv("../data/raw/train.csv")


,Store,Date,Promo,SchoolHoliday,StateHoliday
0,1,2015-07-31,1,1,0
1,2,2015-07-31,1,1,0
2,3,2015-07-31,1,1,0
3,4,2015-07-31,1,1,0
4,5,2015-07-31,1,1,0


In [4]:
features = sales.merge(
    context,
    on=["Store", "Date"],
    how="left"
)

features = features.sort_values(
    ["Store", "Date"]
)

features.head()

,Store,Date,Sales,Promo,SchoolHoliday,StateHoliday
0,1,2013-01-02,5530,0,1,0
1,1,2013-01-03,4327,0,1,0
2,1,2013-01-04,4486,0,1,0
3,1,2013-01-05,4997,0,1,0
4,1,2013-01-07,7176,1,1,0


In [5]:
features.shape

(844392, 6)

In [6]:
features.columns

Index(['Store', 'Date', 'Sales', 'Promo', 'SchoolHoliday', 'StateHoliday'], dtype='str')

In [7]:
features["day_of_week"] = features["Date"].dt.dayofweek

features["day_of_month"] = features["Date"].dt.day

features["month"] = features["Date"].dt.month

features["week_of_year"] = (
    features["Date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

features[
    [
        "Date",
        "day_of_week",
        "day_of_month",
        "month",
        "week_of_year"
    ]
].head()

,Date,day_of_week,day_of_month,month,week_of_year
0,2013-01-02,2,2,1,1
1,2013-01-03,3,3,1,1
2,2013-01-04,4,4,1,1
3,2013-01-05,5,5,1,1
4,2013-01-07,0,7,1,2


In [8]:
grouped_sales = features.groupby("Store")["Sales"]

features["lag_1"] = grouped_sales.shift(1)

features["lag_7"] = grouped_sales.shift(7)

features["lag_14"] = grouped_sales.shift(14)

features["lag_30"] = grouped_sales.shift(30)


In [9]:
features[
    [
        "Store",
        "Date",
        "Sales",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_30"
    ]
].head(15)

,Store,Date,Sales,lag_1,lag_7,lag_14,lag_30
0,1,2013-01-02,5530,NaN,NaN,NaN,NaN
1,1,2013-01-03,4327,5530.0,NaN,NaN,NaN
2,1,2013-01-04,4486,4327.0,NaN,NaN,NaN
3,1,2013-01-05,4997,4486.0,NaN,NaN,NaN
4,1,2013-01-07,7176,4997.0,NaN,NaN,NaN
5,1,2013-01-08,5580,7176.0,NaN,NaN,NaN
6,1,2013-01-09,5471,5580.0,NaN,NaN,NaN
7,1,2013-01-10,4892,5471.0,5530.0,NaN,NaN
8,1,2013-01-11,4881,4892.0,4327.0,NaN,NaN
9,1,2013-01-12,4952,4881.0,4486.0,NaN,NaN


In [10]:
features["rolling_7"] = (
    grouped_sales
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

features["rolling_14"] = (
    grouped_sales
    .transform(
        lambda x: x.shift(1).rolling(14).mean()
    )
)

features["rolling_30"] = (
    grouped_sales
    .transform(
        lambda x: x.shift(1).rolling(30).mean()
    )
)

In [11]:
features[
    [
        "Sales",
        "lag_1",
        "lag_7",
        "rolling_7",
        "rolling_14",
        "rolling_30"
    ]
].head(40)

,Sales,lag_1,lag_7,rolling_7,rolling_14,rolling_30
0,5530,NaN,NaN,NaN,NaN,NaN
1,4327,5530.0,NaN,NaN,NaN,NaN
2,4486,4327.0,NaN,NaN,NaN,NaN
3,4997,4486.0,NaN,NaN,NaN,NaN
4,7176,4997.0,NaN,NaN,NaN,NaN
5,5580,7176.0,NaN,NaN,NaN,NaN
6,5471,5580.0,NaN,NaN,NaN,NaN
7,4892,5471.0,5530.0,5366.714286,NaN,NaN
8,4881,4892.0,4327.0,5275.571429,NaN,NaN
9,4952,4881.0,4486.0,5354.714286,NaN,NaN


In [12]:
feature_columns = [
    "Store",
    "Promo",
    "SchoolHoliday",
    "StateHoliday",
    "day_of_week",
    "day_of_month",
    "month",
    "week_of_year",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_30",
    "rolling_7",
    "rolling_14",
    "rolling_30"
]

In [14]:
model_df = features[
    ["Date", "Sales"] + feature_columns
].copy()

In [16]:
print("Shape:", model_df.shape)

print(
    "Missing values:",
    model_df.isnull().sum().sum()
)

Shape: (810942, 17)
Missing values: 0


In [17]:
model_df = pd.get_dummies(
    model_df,
    columns=["StateHoliday"],
    drop_first=False,
    dtype=int
)

In [18]:
model_df.head()

,Date,Sales,Store,Promo,SchoolHoliday,day_of_week,day_of_month,month,week_of_year,lag_1,lag_7,lag_14,lag_30,rolling_7,rolling_14,rolling_30,StateHoliday_0,StateHoliday_a,StateHoliday_b,StateHoliday_c
0,2013-02-06,6140,1,1,0,2,6,2,6,6049.0,3725.0,5394.0,5530.0,5388.428571,5346.071429,5103.833333,1,0,0,0
1,2013-02-07,5499,1,1,0,3,7,2,6,6140.0,4601.0,5720.0,4327.0,5733.428571,5399.357143,5124.166667,1,0,0,0
2,2013-02-08,5681,1,1,0,4,8,2,6,5499.0,4709.0,5578.0,4486.0,5861.714286,5383.571429,5163.233333,1,0,0,0
3,2013-02-09,5370,1,0,0,5,9,2,6,5681.0,5633.0,5195.0,4997.0,6000.571429,5390.928571,5203.066667,1,0,0,0
4,2013-02-11,4409,1,0,0,0,11,2,7,5370.0,5970.0,5586.0,7176.0,5963.000000,5403.428571,5215.500000,1,0,0,0


In [19]:
model_df.dtypes

Date              datetime64[us]
Sales                      int64
Store                      int64
Promo                      int64
SchoolHoliday              int64
day_of_week                int32
day_of_month               int32
month                      int32
week_of_year               int64
lag_1                    float64
lag_7                    float64
lag_14                   float64
lag_30                   float64
rolling_7                float64
rolling_14               float64
rolling_30               float64
StateHoliday_0             int64
StateHoliday_a             int64
StateHoliday_b             int64
StateHoliday_c             int64
dtype: object

In [20]:
model_df.to_csv(
    "../data/processed/supervised_features.csv",
    index=False
)

print(
    "Saved successfully: "
    "../data/processed/supervised_features.csv"
)

Saved successfully: ../data/processed/supervised_features.csv


In [21]:
print("Rows:", len(model_df))
print("Columns:", len(model_df.columns))

print("\nColumns:")
print(model_df.columns.tolist())

Rows: 810942
Columns: 20

Columns:
['Date', 'Sales', 'Store', 'Promo', 'SchoolHoliday', 'day_of_week', 'day_of_month', 'month', 'week_of_year', 'lag_1', 'lag_7', 'lag_14', 'lag_30', 'rolling_7', 'rolling_14', 'rolling_30', 'StateHoliday_0', 'StateHoliday_a', 'StateHoliday_b', 'StateHoliday_c']


In [22]:
model_df.head()

,Date,Sales,Store,Promo,SchoolHoliday,day_of_week,day_of_month,month,week_of_year,lag_1,lag_7,lag_14,lag_30,rolling_7,rolling_14,rolling_30,StateHoliday_0,StateHoliday_a,StateHoliday_b,StateHoliday_c
0,2013-02-06,6140,1,1,0,2,6,2,6,6049.0,3725.0,5394.0,5530.0,5388.428571,5346.071429,5103.833333,1,0,0,0
1,2013-02-07,5499,1,1,0,3,7,2,6,6140.0,4601.0,5720.0,4327.0,5733.428571,5399.357143,5124.166667,1,0,0,0
2,2013-02-08,5681,1,1,0,4,8,2,6,5499.0,4709.0,5578.0,4486.0,5861.714286,5383.571429,5163.233333,1,0,0,0
3,2013-02-09,5370,1,0,0,5,9,2,6,5681.0,5633.0,5195.0,4997.0,6000.571429,5390.928571,5203.066667,1,0,0,0
4,2013-02-11,4409,1,0,0,0,11,2,7,5370.0,5970.0,5586.0,7176.0,5963.000000,5403.428571,5215.500000,1,0,0,0
